# Module 4.2: Planning Agents and Goal Decomposition

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives

By completing this notebook, you will:
1. Understand goal decomposition strategies for complex tasks
2. Implement a Plan-and-Execute agent architecture
3. Build agents that create, validate, and revise plans
4. Handle dynamic replanning when plans fail
5. Compare different planning approaches (linear, hierarchical, adaptive)

## Prerequisites

- Module 4.1 completed (ReAct agents)
- Understanding of agent loops and tool calling
- Familiarity with state management

## Estimated Time: 80 minutes

---

## 1. Setup & Imports

We'll build planning agents that create comprehensive plans before execution.

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Install required packages
# !pip install anthropic

import json
import os
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum
import anthropic

print("✓ Imports successful")

## 2. Understanding Planning

**Planning** = Breaking down a complex goal into executable steps BEFORE starting execution.

**Why Planning Matters**:
- Handles complex multi-step tasks
- Reduces redundant work
- Makes failure recovery easier
- Enables progress tracking
- Allows validation before expensive operations

**Planning Approaches**:

1. **Linear Planning**: Create a sequential list of steps
   ```
   Goal: Make dinner
   Plan:
   1. Check ingredients
   2. Go shopping if needed
   3. Prep vegetables
   4. Cook meal
   5. Serve
   ```

2. **Hierarchical Planning**: Nested sub-plans
   ```
   Goal: Make dinner
   Plan:
   1. Prepare ingredients
      1.1. Check pantry
      1.2. Make shopping list
      1.3. Go to store
   2. Cook meal
      2.1. Prep vegetables
      2.2. Cook protein
      2.3. Combine ingredients
   ```

3. **Adaptive Planning**: Adjust plan based on observations
   ```
   Plan → Execute Step → Observe → Revise Plan if Needed → Continue
   ```

## 3. Plan Representation

We need to represent plans as structured data for execution and modification.

In [ ]:
# Purpose: Define structured plan representation
# Key Concept: Plans are data structures that can be validated and modified

class StepStatus(Enum):
    """Status of a plan step."""
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    SKIPPED = "skipped"

@dataclass
class PlanStep:
    """A single step in a plan."""
    step_number: int
    description: str
    tool_name: Optional[str] = None
    tool_input: Optional[Dict[str, Any]] = None
    status: StepStatus = StepStatus.PENDING
    result: Optional[Any] = None
    error: Optional[str] = None
    dependencies: List[int] = field(default_factory=list)  # Steps that must complete first
    
    def is_ready(self, completed_steps: set) -> bool:
        """Check if all dependencies are met."""
        return all(dep in completed_steps for dep in self.dependencies)
    
    def __str__(self) -> str:
        status_icon = {
            StepStatus.PENDING: "○",
            StepStatus.IN_PROGRESS: "◐",
            StepStatus.COMPLETED: "●",
            StepStatus.FAILED: "✗",
            StepStatus.SKIPPED: "⊘"
        }
        icon = status_icon.get(self.status, "?")
        return f"{icon} Step {self.step_number}: {self.description}"

@dataclass
class Plan:
    """A complete plan with multiple steps."""
    goal: str
    steps: List[PlanStep] = field(default_factory=list)
    created_at: Optional[str] = None
    
    def add_step(self, step: PlanStep) -> None:
        """Add a step to the plan."""
        self.steps.append(step)
    
    def get_next_steps(self) -> List[PlanStep]:
        """Get steps that are ready to execute."""
        completed = {s.step_number for s in self.steps if s.status == StepStatus.COMPLETED}
        return [
            step for step in self.steps
            if step.status == StepStatus.PENDING and step.is_ready(completed)
        ]
    
    def is_complete(self) -> bool:
        """Check if all steps are completed or skipped."""
        return all(
            s.status in [StepStatus.COMPLETED, StepStatus.SKIPPED]
            for s in self.steps
        )
    
    def has_failed(self) -> bool:
        """Check if any step has failed."""
        return any(s.status == StepStatus.FAILED for s in self.steps)
    
    def display(self) -> str:
        """Generate formatted display of plan."""
        output = f"Goal: {self.goal}\n"
        output += "=" * 60 + "\n"
        for step in self.steps:
            output += str(step) + "\n"
            if step.dependencies:
                output += f"  Dependencies: {step.dependencies}\n"
        return output

# Example plan
example_plan = Plan(goal="Calculate (10 + 5) * 2")
example_plan.add_step(PlanStep(1, "Add 10 + 5", "calculator", {"operation": "add", "a": 10, "b": 5}))
example_plan.add_step(PlanStep(2, "Multiply result by 2", "calculator", dependencies=[1]))
example_plan.steps[0].status = StepStatus.COMPLETED

print(example_plan.display())

## 4. Plan Generation

The planner creates a structured plan from a high-level goal.

In [ ]:
# Purpose: Generate plans from goals using LLM
# Key Concept: LLMs can decompose complex goals into executable steps

class Planner:
    """Creates plans from goals."""
    
    def __init__(self, api_key: Optional[str] = None):
        """
        Initialize planner.
        
        Parameters
        ----------
        api_key : str, optional
            Anthropic API key
        """
        self.client = anthropic.Anthropic(
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY")
        )
    
    def create_plan(
        self,
        goal: str,
        available_tools: List[str],
        context: Optional[str] = None
    ) -> Plan:
        """
        Generate a plan for achieving a goal.
        
        Parameters
        ----------
        goal : str
            The goal to achieve
        available_tools : list of str
            Tools available for execution
        context : str, optional
            Additional context or constraints
        
        Returns
        -------
        Plan
            Generated plan
        """
        # Create planning prompt
        tools_list = "\n".join([f"- {tool}" for tool in available_tools])
        
        prompt = f"""
You are a planning assistant. Create a step-by-step plan to achieve the goal.

GOAL: {goal}

AVAILABLE TOOLS:
{tools_list}

{f'CONTEXT: {context}' if context else ''}

Create a plan with clear, sequential steps. For each step, specify:
1. Step number
2. Description of what to do
3. Which tool to use (if any)
4. Dependencies (which steps must complete first)

Format your response as JSON:
{{
  "steps": [
    {{
      "step_number": 1,
      "description": "First step description",
      "tool": "tool_name",
      "dependencies": []
    }},
    ...
  ]
}}
"""
        
        # Get plan from LLM
        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        
        response_text = response.content[0].text
        
        # Parse JSON from response
        try:
            # Extract JSON from markdown code blocks if present
            import re
            json_match = re.search(r'```json\s*(.+?)\s*```', response_text, re.DOTALL)
            if json_match:
                json_text = json_match.group(1)
            else:
                json_text = response_text
            
            plan_data = json.loads(json_text)
            
            # Create Plan object
            plan = Plan(goal=goal)
            
            for step_data in plan_data.get("steps", []):
                step = PlanStep(
                    step_number=step_data["step_number"],
                    description=step_data["description"],
                    tool_name=step_data.get("tool"),
                    dependencies=step_data.get("dependencies", [])
                )
                plan.add_step(step)
            
            return plan
        
        except (json.JSONDecodeError, KeyError) as e:
            # Fallback: create simple sequential plan
            print(f"Warning: Could not parse plan JSON: {e}")
            print(f"Response was: {response_text[:200]}...")
            
            plan = Plan(goal=goal)
            plan.add_step(PlanStep(1, goal, tool_name=available_tools[0] if available_tools else None))
            return plan

print("Planner class defined")

## 5. Plan Executor

The executor runs a plan, handling tool calls and updating step status.

In [ ]:
# Purpose: Execute plans step by step
# Key Concept: Executors handle the mechanics of running plans

class PlanExecutor:
    """Executes plans with tool calling."""
    
    def __init__(self, tools: Dict[str, callable], verbose: bool = True):
        """
        Initialize executor.
        
        Parameters
        ----------
        tools : dict
            Mapping of tool names to implementations
        verbose : bool
            Print execution progress
        """
        self.tools = tools
        self.verbose = verbose
    
    def execute_step(self, step: PlanStep) -> bool:
        """
        Execute a single plan step.
        
        Parameters
        ----------
        step : PlanStep
            Step to execute
        
        Returns
        -------
        bool
            True if successful, False if failed
        """
        step.status = StepStatus.IN_PROGRESS
        
        if self.verbose:
            print(f"\n→ Executing: {step.description}")
        
        # If no tool specified, mark as completed (informational step)
        if not step.tool_name:
            step.status = StepStatus.COMPLETED
            step.result = {"success": True, "message": "Informational step"}
            return True
        
        # Check if tool exists
        if step.tool_name not in self.tools:
            step.status = StepStatus.FAILED
            step.error = f"Tool '{step.tool_name}' not available"
            if self.verbose:
                print(f"  ✗ Error: {step.error}")
            return False
        
        # Execute tool
        try:
            tool_func = self.tools[step.tool_name]
            result = tool_func(**(step.tool_input or {}))
            
            step.result = result
            
            # Check if tool execution succeeded
            if isinstance(result, dict) and not result.get("success", True):
                step.status = StepStatus.FAILED
                step.error = result.get("error", "Unknown error")
                if self.verbose:
                    print(f"  ✗ Failed: {step.error}")
                return False
            
            step.status = StepStatus.COMPLETED
            if self.verbose:
                print(f"  ✓ Result: {result}")
            return True
        
        except Exception as e:
            step.status = StepStatus.FAILED
            step.error = str(e)
            if self.verbose:
                print(f"  ✗ Exception: {e}")
            return False
    
    def execute_plan(self, plan: Plan, max_failures: int = 3) -> bool:
        """
        Execute a complete plan.
        
        Parameters
        ----------
        plan : Plan
            Plan to execute
        max_failures : int
            Maximum allowed failures before stopping
        
        Returns
        -------
        bool
            True if plan completed successfully
        """
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"Executing Plan: {plan.goal}")
            print("="*60)
        
        failures = 0
        
        while not plan.is_complete() and failures < max_failures:
            # Get next ready steps
            ready_steps = plan.get_next_steps()
            
            if not ready_steps:
                if plan.has_failed():
                    if self.verbose:
                        print("\n✗ Plan execution failed")
                    return False
                else:
                    # No ready steps but not failed - likely dependency issue
                    if self.verbose:
                        print("\n⚠ No ready steps (dependency deadlock?)")
                    return False
            
            # Execute ready steps
            for step in ready_steps:
                success = self.execute_step(step)
                if not success:
                    failures += 1
        
        success = plan.is_complete() and not plan.has_failed()
        
        if self.verbose:
            if success:
                print("\n✓ Plan completed successfully")
            else:
                print("\n✗ Plan execution incomplete or failed")
        
        return success

print("PlanExecutor class defined")

## 6. Complete Planning Agent

Combine planner and executor into a complete planning agent.

In [ ]:
# Purpose: Build complete plan-and-execute agent
# Key Concept: Planning agents separate strategy from execution

class PlanningAgent:
    """Agent that plans then executes."""
    
    def __init__(self, api_key: Optional[str] = None, verbose: bool = True):
        """
        Initialize planning agent.
        
        Parameters
        ----------
        api_key : str, optional
            Anthropic API key
        verbose : bool
            Print progress
        """
        self.planner = Planner(api_key)
        self.tools = {}
        self.verbose = verbose
    
    def register_tool(self, name: str, implementation: callable) -> None:
        """Register a tool."""
        self.tools[name] = implementation
        if self.verbose:
            print(f"✓ Registered: {name}")
    
    def solve(self, goal: str, context: Optional[str] = None) -> Tuple[bool, Plan]:
        """
        Solve a goal by planning and executing.
        
        Parameters
        ----------
        goal : str
            Goal to achieve
        context : str, optional
            Additional context
        
        Returns
        -------
        tuple
            (success, plan)
        """
        # Step 1: Create plan
        if self.verbose:
            print(f"\n{'#'*60}")
            print("PLANNING PHASE")
            print("#"*60)
        
        plan = self.planner.create_plan(
            goal=goal,
            available_tools=list(self.tools.keys()),
            context=context
        )
        
        if self.verbose:
            print("\nGenerated Plan:")
            print(plan.display())
        
        # Step 2: Execute plan
        if self.verbose:
            print(f"\n{'#'*60}")
            print("EXECUTION PHASE")
            print("#"*60)
        
        executor = PlanExecutor(self.tools, verbose=self.verbose)
        success = executor.execute_plan(plan)
        
        return success, plan

print("PlanningAgent class defined")

## 7. Testing Planning Agent

Let's test the planning agent on a complex multi-step task.

In [ ]:
# Purpose: Test planning agent
# Key Concept: Planning enables complex task completion

# Create tools
def calculator(operation: str, a: float, b: float) -> Dict:
    """Basic arithmetic."""
    ops = {"add": a + b, "subtract": a - b, "multiply": a * b, "divide": a / b if b != 0 else None}
    result = ops.get(operation)
    if result is None:
        return {"success": False, "error": "Invalid operation"}
    return {"success": True, "result": result}

_memory = {}

def store(key: str, value: str) -> Dict:
    """Store value in memory."""
    _memory[key] = value
    return {"success": True, "stored": key}

def retrieve(key: str) -> Dict:
    """Retrieve value from memory."""
    if key in _memory:
        return {"success": True, "value": _memory[key]}
    return {"success": False, "error": f"Key not found: {key}"}

# Create agent
agent = PlanningAgent(verbose=True)
agent.register_tool("calculator", calculator)
agent.register_tool("store", store)
agent.register_tool("retrieve", retrieve)

print("\nAgent ready")

In [ ]:
# Test complex goal
goal = "Calculate 15 * 4, store the result as 'total', then calculate 20% of that total."

success, plan = agent.solve(goal)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Success: {success}")
print(f"\nFinal Plan State:")
print(plan.display())

## Hands-On Exercises

Build and enhance planning agents.

### Exercise 4.2.1: Plan Validator

**Task**: Create a function that validates a plan before execution.

**Expected Output**: Function returns dict with validation results (is_valid, issues).

**Validation checks**:
- All referenced tools exist
- No circular dependencies
- Dependencies reference valid step numbers

**Hints**:
<details>
<summary>Hint 1</summary>
Check step.tool_name against available_tools list.
</details>

<details>
<summary>Hint 2</summary>
For circular dependencies, use graph traversal or check if a step depends on itself transitively.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def validate_plan(plan: Plan, available_tools: List[str]) -> Dict[str, Any]:
    """
    Validate a plan before execution.
    
    Parameters
    ----------
    plan : Plan
        Plan to validate
    available_tools : list of str
        Available tool names
    
    Returns
    -------
    dict
        Validation results with is_valid and issues
    """
    # TODO: Implement plan validation
    pass

exercise_4_2_1_answer = validate_plan  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_2_1():
    # Valid plan
    valid_plan = Plan("test")
    valid_plan.add_step(PlanStep(1, "step 1", "tool_a"))
    valid_plan.add_step(PlanStep(2, "step 2", "tool_b", dependencies=[1]))
    
    result = exercise_4_2_1_answer(valid_plan, ["tool_a", "tool_b"])
    assert isinstance(result, dict), "Should return dict"
    assert "is_valid" in result, "Should have is_valid field"
    assert result["is_valid"] == True, "Valid plan should pass validation"
    
    # Invalid plan (unknown tool)
    invalid_plan = Plan("test")
    invalid_plan.add_step(PlanStep(1, "step 1", "unknown_tool"))
    
    result = exercise_4_2_1_answer(invalid_plan, ["tool_a"])
    assert result["is_valid"] == False, "Plan with unknown tool should fail"
    assert "issues" in result, "Should list issues"
    
    print("✓ Exercise 4.2.1 passed!")

test_exercise_4_2_1()

### Exercise 4.2.2: Progress Tracker

**Task**: Create a function that calculates plan progress.

**Expected Output**: Dict with: percent_complete, completed_steps, total_steps, estimated_remaining.

**Hints**:
<details>
<summary>Hint 1</summary>
Count steps by status: completed / total * 100.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def track_progress(plan: Plan) -> Dict[str, Any]:
    """
    Calculate plan execution progress.
    
    Parameters
    ----------
    plan : Plan
        Plan to track
    
    Returns
    -------
    dict
        Progress metrics
    """
    # TODO: Calculate progress metrics
    pass

exercise_4_2_2_answer = track_progress  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_2_2():
    test_plan = Plan("test")
    test_plan.add_step(PlanStep(1, "s1"))
    test_plan.add_step(PlanStep(2, "s2"))
    test_plan.add_step(PlanStep(3, "s3"))
    test_plan.steps[0].status = StepStatus.COMPLETED
    
    result = exercise_4_2_2_answer(test_plan)
    
    assert isinstance(result, dict), "Should return dict"
    assert "percent_complete" in result or "progress" in result, "Should have progress percentage"
    assert "completed_steps" in result, "Should count completed steps"
    assert result["completed_steps"] == 1, "Should have 1 completed step"
    assert "total_steps" in result, "Should count total steps"
    assert result["total_steps"] == 3, "Should have 3 total steps"
    
    print("✓ Exercise 4.2.2 passed!")

test_exercise_4_2_2()

### Exercise 4.2.3: Adaptive Replanning

**Task**: When a step fails, create a function that generates an alternative plan.

**Expected Output**: Function that takes failed step and returns modified plan.

**Strategy**: Skip failed step or replace it with alternative approach.

**Hints**:
<details>
<summary>Hint 1</summary>
Mark failed step as SKIPPED and remove it from dependencies of later steps.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def replan_after_failure(plan: Plan, failed_step_number: int) -> Plan:
    """
    Modify plan after step failure.
    
    Parameters
    ----------
    plan : Plan
        Original plan
    failed_step_number : int
        Step number that failed
    
    Returns
    -------
    Plan
        Modified plan
    """
    # TODO: Implement replanning logic
    pass

exercise_4_2_3_answer = replan_after_failure  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_2_3():
    test_plan = Plan("test")
    test_plan.add_step(PlanStep(1, "s1"))
    test_plan.add_step(PlanStep(2, "s2"))
    test_plan.add_step(PlanStep(3, "s3", dependencies=[1, 2]))
    test_plan.steps[1].status = StepStatus.FAILED
    
    new_plan = exercise_4_2_3_answer(test_plan, 2)
    
    assert isinstance(new_plan, Plan), "Should return Plan"
    # Check that step 2 is handled (skipped or modified)
    step_2 = [s for s in new_plan.steps if s.step_number == 2][0]
    assert step_2.status in [StepStatus.FAILED, StepStatus.SKIPPED], "Failed step should be marked"
    
    print("✓ Exercise 4.2.3 passed!")

test_exercise_4_2_3()

## Solutions

In [ ]:
# SOLUTION 4.2.1: Plan Validator

def validate_plan_solution(plan: Plan, available_tools: List[str]) -> Dict[str, Any]:
    issues = []
    
    step_numbers = {s.step_number for s in plan.steps}
    
    for step in plan.steps:
        # Check tool exists
        if step.tool_name and step.tool_name not in available_tools:
            issues.append(f"Step {step.step_number}: Unknown tool '{step.tool_name}'")
        
        # Check dependencies are valid
        for dep in step.dependencies:
            if dep not in step_numbers:
                issues.append(f"Step {step.step_number}: Invalid dependency {dep}")
            if dep >= step.step_number:
                issues.append(f"Step {step.step_number}: Circular/forward dependency {dep}")
    
    return {
        "is_valid": len(issues) == 0,
        "issues": issues
    }

# SOLUTION 4.2.2: Progress Tracker

def track_progress_solution(plan: Plan) -> Dict[str, Any]:
    total = len(plan.steps)
    completed = sum(1 for s in plan.steps if s.status == StepStatus.COMPLETED)
    pending = sum(1 for s in plan.steps if s.status == StepStatus.PENDING)
    
    return {
        "total_steps": total,
        "completed_steps": completed,
        "pending_steps": pending,
        "percent_complete": (completed / total * 100) if total > 0 else 0,
        "estimated_remaining": pending
    }

# SOLUTION 4.2.3: Adaptive Replanning

def replan_after_failure_solution(plan: Plan, failed_step_number: int) -> Plan:
    # Mark failed step as skipped
    for step in plan.steps:
        if step.step_number == failed_step_number:
            step.status = StepStatus.SKIPPED
        
        # Remove failed step from dependencies
        if failed_step_number in step.dependencies:
            step.dependencies = [d for d in step.dependencies if d != failed_step_number]
    
    return plan

## Summary

**Key Takeaways**:

1. **Planning First**: Separating planning from execution enables validation and optimization
2. **Structured Plans**: Plans as data structures can be analyzed, modified, and debugged
3. **Dependency Management**: Explicit dependencies enable parallel execution and failure recovery
4. **Progress Tracking**: Monitoring plan state enables user feedback and adaptive behavior
5. **Replanning**: Dynamic plan modification handles failures and changing conditions

**What's Next**:
- Module 4.3: Self-reflection and error correction
- Module 5: Memory and long-term state management
- Module 6: Multi-agent systems

**Additional Resources**:
- [Plan-and-Execute Agents (LangChain)](https://python.langchain.com/docs/use_cases/more/agents/autonomous_agents/plan_and_execute)
- [Hierarchical Planning](https://en.wikipedia.org/wiki/Hierarchical_task_network)

---

You've built a sophisticated planning agent that decomposes complex goals into executable steps!